In [4]:
# Student Information
import numpy as np
import pandas as pd
from pathlib import Path
import os

STUDENT_ID = "TUPM-26-1129"  
LAST_NAME = "Espejo"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(digit) for digit in id_digits)
NAME_LENGTH = len(LAST_NAME)

# Keep this formula consistent with the main laboratory activity.
RANDOM_SEED = ID_SUM + SEED_DIGIT + NAME_LENGTH

print("STUDENT CONFIGURATION")
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)

STUDENT CONFIGURATION
Seed Digit: 9
ID Sum: 21
Name Length: 6
Random Seed: 36


In [5]:

CURRENT_DIR = Path.cwd()
PROJECT_DIR = None

# Locate the main project folder
for folder in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (
        (folder / "data" / "secom.data").exists()
        and (folder / "data" / "secom_labels.data").exists()
    ):
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Cannot find SECOM files. Place secom.data and "
        "secom_labels.data inside the data folder."
    )

DATA_PATH = PROJECT_DIR / "data" / "secom.data"
LABEL_PATH = PROJECT_DIR / "data" / "secom_labels.data"

OUTPUT_DIR = (
    PROJECT_DIR
    / "Exercise_3_Statistical_Robustness"
    / "output"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load measurements and labels
measurements = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None
)

labels = pd.read_csv(
    LABEL_PATH,
    sep=r"\s+",
    header=None,
    usecols=[0]
)

measurements.columns = [
    f"Feature_{i+1}" for i in range(measurements.shape[1])
]

labels.columns = ["Outcome"]

# Check dataset row counts
if len(measurements) != len(labels):
    raise ValueError("Measurement and label row counts do not match.")

# Combine datasets while preserving original record identifiers
data = measurements.copy()

data.insert(
    0,
    "Record_ID",
    np.arange(len(data))
)

data["Outcome"] = pd.to_numeric(
    labels["Outcome"],
    errors="coerce"
)

# Use the SECOM convention: -1 = Fail and 1 = Pass
data = data[data["Outcome"].isin([-1, 1])].copy()

data["Outcome_Name"] = data["Outcome"].map({
    -1: "Fail",
    1: "Pass"
})

feature_columns = [
    column for column in measurements.columns
]

data[feature_columns] = data[feature_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("DATASET INFORMATION")
print("Total Records:", len(data))
print("Total Features:", len(feature_columns))
print("Pass Records:", (data["Outcome_Name"] == "Pass").sum())
print("Fail Records:", (data["Outcome_Name"] == "Fail").sum())

data.head()


DATASET INFORMATION
Total Records: 1567
Total Features: 590
Pass Records: 104
Fail Records: 1463


,Record_ID,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Feature_590,Outcome,Outcome_Name
0,0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1,Fail
1,1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1,Fail
2,2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,Pass
3,3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1,Fail
4,4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1,Fail


In [6]:

# Identify eligible features
eligible_features_3 = []

for feature in feature_columns:

    fail_observed = data.loc[
        data["Outcome_Name"] == "Fail",
        feature
    ].notna().sum()

    pass_observed = data.loc[
        data["Outcome_Name"] == "Pass",
        feature
    ].notna().sum()

    if fail_observed >= 10 and pass_observed >= 10:
        eligible_features_3.append(feature)

# Check availability
if len(eligible_features_3) < 5:
    raise ValueError(
        "Fewer than five features have at least "
        "10 observations in both groups."
    )

# Select five features reproducibly
rng3 = np.random.default_rng(RANDOM_SEED + 2)

selected_features_3 = sorted(
    rng3.choice(
        eligible_features_3,
        size=5,
        replace=False
    ).tolist()
)

print("FEATURE SELECTION RESULTS")
print("--------------------------------")

print("Eligible Features:", len(eligible_features_3))

print("\nSelected Five Features:")

for feature in selected_features_3:
    print("-", feature)

print("\nNumber Selected:", len(selected_features_3))


FEATURE SELECTION RESULTS
--------------------------------
Eligible Features: 586

Selected Five Features:
- Feature_146
- Feature_147
- Feature_191
- Feature_231
- Feature_286

Number Selected: 5


In [7]:

# Bootstrap configuration
BOOTSTRAP_REPETITIONS = 200 + (10 * SEED_DIGIT)

BOOTSTRAP_SEED = RANDOM_SEED + 3

print("BOOTSTRAP CONFIGURATION")
print("Repetitions:", BOOTSTRAP_REPETITIONS)
print("Bootstrap Seed:", BOOTSTRAP_SEED)


# Calculate Fail mean minus Pass mean
def mean_difference(fail_values, pass_values):

    return float(
        np.mean(fail_values) - np.mean(pass_values)
    )


# Perform stratified bootstrap resampling
def bootstrap_differences(
    fail_values,
    pass_values,
    repetitions,
    seed
):

    rng = np.random.default_rng(seed)

    differences = np.empty(
        repetitions,
        dtype=float
    )

    for repetition in range(repetitions):

        fail_resample = rng.choice(
            fail_values,
            size=len(fail_values),
            replace=True
        )

        pass_resample = rng.choice(
            pass_values,
            size=len(pass_values),
            replace=True
        )

        differences[repetition] = mean_difference(
            fail_resample,
            pass_resample
        )

    return differences


BOOTSTRAP CONFIGURATION
Repetitions: 290
Bootstrap Seed: 39


In [8]:

sensitivity_rows = []

largest_change = 0.0

for feature_index, feature in enumerate(selected_features_3):

    # --------------------------------
    # STRATEGY A: AVAILABLE-CASE
    # --------------------------------

    fail_original = data.loc[
        data["Outcome_Name"] == "Fail",
        feature
    ].dropna().to_numpy(dtype=float)

    pass_original = data.loc[
        data["Outcome_Name"] == "Pass",
        feature
    ].dropna().to_numpy(dtype=float)

    available_difference = mean_difference(
        fail_original,
        pass_original
    )

    # Bootstrap for available-case analysis
    available_bootstrap = bootstrap_differences(
        fail_original,
        pass_original,
        BOOTSTRAP_REPETITIONS,
        BOOTSTRAP_SEED + feature_index
    )

    available_ci_low, available_ci_high = np.percentile(
        available_bootstrap,
        [2.5, 97.5]
    )

    # --------------------------------
    # STRATEGY B: MEDIAN IMPUTATION
    # --------------------------------

    full_median = data[feature].median()

    fail_imputed = data.loc[
        data["Outcome_Name"] == "Fail",
        feature
    ].fillna(full_median).to_numpy(dtype=float)

    pass_imputed = data.loc[
        data["Outcome_Name"] == "Pass",
        feature
    ].fillna(full_median).to_numpy(dtype=float)

    imputed_difference = mean_difference(
        fail_imputed,
        pass_imputed
    )

    # Bootstrap for median-imputed data
    imputed_bootstrap = bootstrap_differences(
        fail_imputed,
        pass_imputed,
        BOOTSTRAP_REPETITIONS,
        BOOTSTRAP_SEED + 100 + feature_index
    )

    imputed_ci_low, imputed_ci_high = np.percentile(
        imputed_bootstrap,
        [2.5, 97.5]
    )

    # --------------------------------
    # SENSITIVITY COMPARISON
    # --------------------------------

    change = abs(
        imputed_difference - available_difference
    )

    largest_change = max(
        largest_change,
        change
    )

    sensitivity_rows.append({

        "Feature": feature,

        "Available_Case_Mean_Difference":
            available_difference,

        "Available_Case_CI_95_Lower":
            available_ci_low,

        "Available_Case_CI_95_Upper":
            available_ci_high,

        "Median_Imputation_Mean_Difference":
            imputed_difference,

        "Median_Imputation_CI_95_Lower":
            imputed_ci_low,

        "Median_Imputation_CI_95_Upper":
            imputed_ci_high,

        "Absolute_Change_in_Mean_Difference":
            change,

        "Direction_Consistent":
            np.sign(available_difference)
            == np.sign(imputed_difference)
    })

print("Available-case and median-imputation analyses completed.")
print("Features analyzed:", len(sensitivity_rows))
print("Bootstrap repetitions:", BOOTSTRAP_REPETITIONS)


Available-case and median-imputation analyses completed.
Features analyzed: 5
Bootstrap repetitions: 290


In [9]:

# Convert results into a DataFrame
sensitivity = pd.DataFrame(sensitivity_rows)

# Rank features based on absolute mean differences
sensitivity["Available_Case_Rank"] = (
    sensitivity["Available_Case_Mean_Difference"]
    .abs()
    .rank(ascending=False, method="min")
)

sensitivity["Median_Imputation_Rank"] = (
    sensitivity["Median_Imputation_Mean_Difference"]
    .abs()
    .rank(ascending=False, method="min")
)

# Display comparison
print("STATISTICAL ROBUSTNESS COMPARISON")
display(sensitivity)

# Export CSV file
sensitivity.to_csv(
    OUTPUT_DIR / "robustness_comparison.csv",
    index=False
)

print("CSV file successfully saved!")
print(OUTPUT_DIR / "robustness_comparison.csv")


STATISTICAL ROBUSTNESS COMPARISON


,Feature,Available_Case_Mean_Difference,Available_Case_CI_95_Lower,Available_Case_CI_95_Upper,Median_Imputation_Mean_Difference,Median_Imputation_CI_95_Lower,Median_Imputation_CI_95_Upper,Absolute_Change_in_Mean_Difference,Direction_Consistent,Available_Case_Rank,Median_Imputation_Rank
0,Feature_146,0.004678,0.000954,0.008098,0.004671,0.001375,0.007817,0.000007,True,2.0,2.0
1,Feature_147,0.003814,0.000717,0.006931,0.003807,0.000374,0.007273,0.000007,True,3.0,3.0
2,Feature_191,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,4.0,4.0
3,Feature_231,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,4.0,4.0
4,Feature_286,0.047707,-0.164205,0.222160,0.047199,-0.154410,0.231328,0.000508,True,1.0,1.0


CSV file successfully saved!
c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_3_Statistical_Robustness\output\robustness_comparison.csv


In [10]:

# Select the first feature for reproducibility testing
check_feature = selected_features_3[0]

check_fail = data.loc[
    data["Outcome_Name"] == "Fail",
    check_feature
].dropna().to_numpy(dtype=float)

check_pass = data.loc[
    data["Outcome_Name"] == "Pass",
    check_feature
].dropna().to_numpy(dtype=float)

# First bootstrap execution
check_a = bootstrap_differences(
    check_fail,
    check_pass,
    BOOTSTRAP_REPETITIONS,
    BOOTSTRAP_SEED
)

# Second bootstrap execution using the SAME seed
check_b = bootstrap_differences(
    check_fail,
    check_pass,
    BOOTSTRAP_REPETITIONS,
    BOOTSTRAP_SEED
)

# Compare both results
reproducibility_verified = np.array_equal(
    check_a,
    check_b
)

print("REPRODUCIBILITY CHECK")
print("-----------------------------")
print("First Bootstrap:", check_a[:5])
print("Second Bootstrap:", check_b[:5])
print("Reproducibility Verified:", reproducibility_verified)


REPRODUCIBILITY CHECK
-----------------------------
First Bootstrap: [0.00516367 0.00679057 0.00612111 0.00350041 0.00588767]
Second Bootstrap: [0.00516367 0.00679057 0.00612111 0.00350041 0.00588767]
Reproducibility Verified: True


In [11]:

interpretation_lines = [

    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT",

    f"Selected Features: {', '.join(selected_features_3)}",

    f"Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}",

    f"Random Seed: {BOOTSTRAP_SEED}",

    f"Reproducibility Verified: {reproducibility_verified}",

    "",

    "INTERPRETATION:"
]

for _, row in sensitivity.iterrows():

    direction = (
        "Same direction"
        if row["Direction_Consistent"]
        else "Different direction"
    )

    interpretation_lines.append(

        f"- {row['Feature']}: {direction}. "
        f"Absolute change in mean difference: "
        f"{row['Absolute_Change_in_Mean_Difference']:.6g}"
    )


interpretation_lines.append(

    f"\nLargest change in mean difference: "
    f"{largest_change:.6g}"
)

# Save text report
report_path = OUTPUT_DIR / "sensitivity_report.txt"

report_path.write_text(
    "\n".join(interpretation_lines),
    encoding="utf-8"
)

print("SENSITIVITY REPORT")
print("-----------------------------")

for line in interpretation_lines:
    print(line)

print("\nReport successfully saved!")
print(report_path)


SENSITIVITY REPORT
-----------------------------
SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT
Selected Features: Feature_146, Feature_147, Feature_191, Feature_231, Feature_286
Bootstrap Repetitions: 290
Random Seed: 39
Reproducibility Verified: True

INTERPRETATION:
- Feature_146: Same direction. Absolute change in mean difference: 7.28955e-06
- Feature_147: Same direction. Absolute change in mean difference: 7.19551e-06
- Feature_191: Same direction. Absolute change in mean difference: 0
- Feature_231: Same direction. Absolute change in mean difference: 0
- Feature_286: Same direction. Absolute change in mean difference: 0.00050817

Largest change in mean difference: 0.00050817

Report successfully saved!
c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_3_Statistical_Robustness\output\sensitivity_report.txt


In [12]:

# Determine how many features maintained the same direction
consistent_features = int(
    sensitivity["Direction_Consistent"].sum()
)

# Determine the feature with the largest sensitivity change
largest_change_feature = sensitivity.loc[
    sensitivity["Absolute_Change_in_Mean_Difference"].idxmax(),
    "Feature"
]

# Create assessment table
assessment = pd.DataFrame({

    "Output Parameter": [

        "Selected Features",

        "Bootstrap Repetitions",

        "Features with Consistent Direction",

        "Largest Sensitivity Change",

        "Feature with Largest Change",

        "Reproducibility Verified"
    ],

    "Observed Value": [

        ", ".join(selected_features_3),

        BOOTSTRAP_REPETITIONS,

        f"{consistent_features} out of 5",

        f"{largest_change:.6f}",

        largest_change_feature,

        reproducibility_verified
    ]
})

print("=" * 65)
print("EXERCISE 3: ASSESSMENT DATA")
print("=" * 65)

display(assessment)

print("\nGENERATED OUTPUT FILES")
print("-----------------------------")
print("1. robustness_comparison.csv")
print("2. sensitivity_report.txt")

print("\nOutput Folder:", OUTPUT_DIR)

print("=" * 65)


EXERCISE 3: ASSESSMENT DATA


,Output Parameter,Observed Value
0,Selected Features,"Feature_146, Feature_147, Feature_191, Feature..."
1,Bootstrap Repetitions,290
2,Features with Consistent Direction,5 out of 5
3,Largest Sensitivity Change,0.000508
4,Feature with Largest Change,Feature_286
5,Reproducibility Verified,True



GENERATED OUTPUT FILES
-----------------------------
1. robustness_comparison.csv
2. sensitivity_report.txt

Output Folder: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_3_Statistical_Robustness\output
